# Notebook 04 Inference Chat RAG Tanpa Thinking

Notebook ini khusus untuk inference/chat interaktif. Batch ground truth, scoring, dan plot evaluasi ada di notebook 05.

In [ ]:
# Jalankan sekali di environment baru bila package belum ada
# !pip install -q chromadb rank_bm25 sentence-transformers transformers accelerate bitsandbytes

In [ ]:
import os, re, json, time, pickle, hashlib
from pathlib import Path
from typing import Dict, List, Any

import numpy as np
import torch
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

DATA_PATH = Path("../data/processed_chunks_ringan_pasal_chroma_ready.json")
CHROMA_DB_DIR = Path("../data/chroma_db")
COLLECTION_NAME = "hukum_ketenagakerjaan"
BM25_PATH = Path("../data/bm25_index.pkl")

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
USE_RERANKER = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device", DEVICE)
print("data", DATA_PATH.resolve())

In [ ]:
def tokenize_for_bm25(text: str) -> List[str]:
    return re.findall(r"[a-zA-Z0-9_]+|[\u00C0-\u024F\u1E00-\u1EFF]+|[\w]+", str(text).lower())


def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text


def load_chunks(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Chunk tidak ditemukan: {path}. Jalankan notebook 01 untuk embedding dan indexing dulu.")
    chunks = json.loads(path.read_text(encoding="utf-8"))
    for i, c in enumerate(chunks[:10]):
        for k in ["id", "text", "display_text", "embedding_text", "citation_text", "metadata"]:
            if k not in c:
                raise ValueError(f"Schema chunk belum sesuai. Field hilang: {k} pada index {i}")
    return chunks

chunks = load_chunks(DATA_PATH)
id_to_chunk = {c["id"]: c for c in chunks}
print("chunks", len(chunks))

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))
collection = client.get_collection(COLLECTION_NAME)
print("chroma count", collection.count())
if collection.count() != len(chunks):
    raise ValueError(f"Chroma count {collection.count()} tidak sama dengan chunks {len(chunks)}. Jalankan notebook 01 dulu.")

bm25 = None
bm25_ids = []
if BM25_PATH.exists():
    with BM25_PATH.open("rb") as f:
        payload = pickle.load(f)
    bm25 = payload.get("bm25")
    bm25_ids = payload.get("ids", [])
    print("bm25", len(bm25_ids))
else:
    print("BM25 tidak ditemukan. Dense retrieval tetap berjalan.")

In [ ]:
MAX_CONTEXT_DOCS = 4
MIN_QUERY_TERM_OVERLAP = 1

LEGAL_QUERY_EXPANSIONS = [
    {
        "triggers": ["pelanggaran berat", "mendesak", "bersifat mendesak"],
        "expansion": (
            "pelanggaran bersifat mendesak uang pisah uang penggantian hak "
            "tidak mendapat pesangon tidak mendapat uang penghargaan masa kerja"
        ),
    },
    {
        "triggers": ["surat peringatan", "sp pertama", "sp kedua", "sp ketiga"],
        "expansion": (
            "pelanggaran ketentuan surat peringatan pertama kedua ketiga "
            "pesangon nol koma lima uang penghargaan masa kerja uang penggantian hak"
        ),
    },
    {
        "triggers": ["pesangon", "phk", "pemutusan hubungan kerja"],
        "expansion": (
            "pemutusan hubungan kerja uang pesangon uang penghargaan masa kerja "
            "uang penggantian hak hak akibat pemutusan hubungan kerja"
        ),
    },
    {
        "triggers": ["pkwt", "kontrak"],
        "expansion": (
            "perjanjian kerja waktu tertentu kompensasi pkwt "
            "jangka waktu perpanjangan pembaruan"
        ),
    },
    {
        "triggers": ["alih daya", "outsourcing", "outsourced"],
        "expansion": (
            "alih daya perusahaan alih daya pekerja buruh "
            "hubungan kerja perlindungan upah kesejahteraan"
        ),
    },
    {
        "triggers": ["upah", "gaji", "tunjangan"],
        "expansion": (
            "upah gaji tunjangan tetap struktur skala upah "
            "upah minimum pembayaran upah"
        ),
    },
]

KETENAGAKERJAAN_POSITIVE_KEYWORDS = {
    "ketenagakerjaan", "tenaga kerja", "pekerja", "buruh", "hubungan kerja",
    "perjanjian kerja", "pemutusan hubungan kerja", "phk", "pesangon",
    "upah", "pengupahan", "pensiun", "jaminan sosial", "jaminan kerja",
    "alih daya", "outsourcing", "pkwt", "pkwtt", "serikat pekerja",
    "pengusaha", "perusahaan alih daya", "perlindungan pekerja",
    "waktu kerja", "cuti", "k3", "keselamatan kerja", "cipta kerja",
}

KETENAGAKERJAAN_NEGATIVE_KEYWORDS = {
    "perizinan berusaha", "oss", "risiko usaha", "izin usaha", "nib",
    "investasi", "badan usaha", "penyelenggaraan usaha", "sistem perizinan",
    "rba", "risk based approach", "sektor usaha", "kbli",
    "penyelenggaraan pemerintahan", "administrasi pemerintahan",
    "sop administrasi", "pelayanan publik",
}

KETENAGAKERJAAN_KNOWN_REGS = {
    ("pp", "35"), ("pp", "36"), ("pp", "34"), ("pp", "45"),
    ("uu", "13"), ("uu", "6"), ("uu", "24"), ("uu", "1"), ("uu", "21"),
    ("permen", "5"), ("permen", "6"),
}


def normalize_reg_type(value: str) -> str:
    value = str(value or "").lower()
    if "undang" in value or value == "uu":
        return "uu"
    if "pemerintah" in value or value == "pp":
        return "pp"
    if "presiden" in value or "perpres" in value:
        return "perpres"
    if "menteri" in value or "permen" in value:
        return "permen"
    return value


def is_ketenagakerjaan_doc(meta: Dict[str, Any]) -> bool:
    tentang = str(meta.get("tentang", "") or "").lower()
    bab_title = str(meta.get("bab_title", "") or "").lower()
    bagian_title = str(meta.get("bagian_title", "") or "").lower()
    haystack = f"{tentang} {bab_title} {bagian_title}"
    if any(neg in haystack for neg in KETENAGAKERJAAN_NEGATIVE_KEYWORDS):
        return False
    reg_type = normalize_reg_type(meta.get("regulation_type", ""))
    nomor = str(meta.get("nomor", "") or "").strip()
    if (reg_type, nomor) in KETENAGAKERJAAN_KNOWN_REGS:
        return True
    if any(pos in haystack for pos in KETENAGAKERJAAN_POSITIVE_KEYWORDS):
        return True
    return not tentang.strip()


def expand_query_terms(query: str) -> str:
    q = query.lower()
    expansions = []
    for rule in LEGAL_QUERY_EXPANSIONS:
        if any(trigger in q for trigger in rule["triggers"]):
            expansions.append(rule["expansion"])
    return normalize_text(" ".join([query] + expansions))


def retrieval_queries(query: str) -> List[str]:
    expanded = expand_query_terms(query)
    return [query, expanded] if expanded != query else [query]


def dense_search(query: str, fetch_k: int = 40) -> List[Dict[str, Any]]:
    q_emb = embedding_model.encode(["query: " + query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)[0].tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=fetch_k, include=["documents", "metadatas", "distances"])
    hits = []
    for doc_id, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"id": doc_id, "text": doc, "metadata": meta or {}, "dense_distance": float(dist), "source": "dense"})
    return hits


def bm25_search(query: str, fetch_k: int = 40) -> List[Dict[str, Any]]:
    if bm25 is None or not bm25_ids:
        return []
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = np.argsort(scores)[::-1][:fetch_k]
    ids = [bm25_ids[i] for i in order if scores[i] > 0]
    if not ids:
        return []
    got = collection.get(ids=ids, include=["documents", "metadatas"])
    lookup = {doc_id: (doc, meta) for doc_id, doc, meta in zip(got["ids"], got["documents"], got["metadatas"])}
    hits = []
    for i in order:
        doc_id = bm25_ids[i]
        if scores[i] <= 0 or doc_id not in lookup:
            continue
        doc, meta = lookup[doc_id]
        hits.append({"id": doc_id, "text": doc, "metadata": meta or {}, "bm25_score": float(scores[i]), "source": "bm25"})
    return hits


def rrf_fuse(result_sets: List[List[Dict[str, Any]]], weights=None, rrf_k: int = 60) -> List[Dict[str, Any]]:
    weights = weights or [1.0] * len(result_sets)
    fused = {}
    for hits, weight in zip(result_sets, weights):
        for rank, hit in enumerate(hits, 1):
            item = fused.setdefault(hit["id"], {"score": 0.0, "hit": hit})
            item["score"] += weight / (rrf_k + rank)
            item["hit"].update({k: v for k, v in hit.items() if k not in item["hit"]})
    out = []
    for item in sorted(fused.values(), key=lambda x: x["score"], reverse=True):
        hit = item["hit"]
        hit["rrf_score"] = float(item["score"])
        out.append(hit)
    return out


def lex_posterior_score(hit: Dict[str, Any]) -> float:
    meta = hit.get("metadata", {})
    score = float(hit.get("rrf_score", 0.0))
    try:
        year = int(meta.get("publication_year") or meta.get("year") or 0)
    except Exception:
        year = 0
    try:
        hierarchy = int(meta.get("regulation_hierarchy") or 99)
    except Exception:
        hierarchy = 99
    if str(meta.get("active_status", "")).lower() == "berlaku":
        score += 0.030
    if is_ketenagakerjaan_doc(meta):
        score += min(max(year - 2000, 0), 40) * 0.001
    else:
        score -= 0.100
    score += max(0, 6 - hierarchy) * 0.003
    if meta.get("quality_status") == "needs_review":
        score -= 0.010
    return score


def dedupe_legal_hits(hits: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    ranked = sorted(hits, key=lex_posterior_score, reverse=True)
    seen, out = set(), []
    for hit in ranked:
        meta = hit.get("metadata", {})
        if not is_ketenagakerjaan_doc(meta):
            continue
        key = (meta.get("source_file", ""), meta.get("pasal_id", ""), meta.get("chunk_kind", ""), meta.get("chunk_index", ""))
        if key in seen:
            continue
        seen.add(key)
        hit["final_score"] = lex_posterior_score(hit)
        out.append(hit)
        if len(out) >= k:
            break
    return out


reranker = None
if USE_RERANKER:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(RERANKER_MODEL_NAME, device=DEVICE)
    print("Reranker aktif:", RERANKER_MODEL_NAME)
else:
    print("Reranker neural nonaktif. Retrieval memakai RRF dense+BM25 + dedupe legal.")


def rerank_documents(query: str, docs: List[Dict[str, Any]], k: int = 8) -> List[Dict[str, Any]]:
    if reranker is None or not docs:
        return docs[:k]
    scores = reranker.predict([(query, d["text"]) for d in docs])
    for d, s in zip(docs, scores):
        d["rerank_score"] = float(s)
    return sorted(docs, key=lambda x: x.get("rerank_score", 0.0), reverse=True)[:k]


def retrieve_documents(query: str, k: int = 8, fetch_k: int = 40, use_bm25: bool = True) -> List[Dict[str, Any]]:
    dense_hits = dense_search(query, fetch_k=fetch_k)
    sparse_hits = bm25_search(query, fetch_k=fetch_k) if use_bm25 else []
    fused = rrf_fuse([dense_hits, sparse_hits], weights=[1.0, 0.7]) if sparse_hits else dense_hits
    return dedupe_legal_hits(fused, k=k)


def query_terms(query: str) -> set[str]:
    stopwords = {
        "yang", "dan", "atau", "karena", "dengan", "untuk", "pada", "dalam",
        "jika", "maka", "dari", "berapa", "apakah", "bagaimana", "dimana",
        "kapan", "siapa", "pekerja", "buruh", "pengusaha", "perusahaan", "hak",
        "nya", "itu", "ini", "ada", "dapat", "bisa", "oleh", "ke", "di", "atas",
    }
    return {t for t in re.findall(r"[a-zA-Z0-9]+", query.lower()) if len(t) > 2 and t not in stopwords}


def doc_relevance_score(query: str, doc: Dict[str, Any]) -> float:
    terms = query_terms(expand_query_terms(query))
    if not terms:
        return 1.0
    meta = doc.get("metadata", {})
    haystack = " ".join([
        str(doc.get("text", "")), str(meta.get("citation_text", "")), str(meta.get("regulation_type", "")),
        str(meta.get("nomor", "")), str(meta.get("pasal_id", "")), str(meta.get("bab_title", "")), str(meta.get("bagian_title", "")),
    ]).lower()
    score = sum(1 for term in terms if term in haystack) / max(len(terms), 1)
    original_terms = query_terms(query)
    specific_terms = {t for t in original_terms if t not in {"pesangon", "phk", "pemutusan", "hubungan", "kerja"}}
    if specific_terms and not any(term in haystack for term in specific_terms):
        score -= 0.25
    if "final_score" in doc:
        score += min(float(doc.get("final_score", 0.0)), 1.0) * 0.05
    elif "rrf_score" in doc:
        score += min(float(doc.get("rrf_score", 0.0)), 1.0) * 0.05
    return score


def has_original_specific_overlap(query: str, doc: Dict[str, Any]) -> bool:
    original_terms = query_terms(query)
    specific_terms = {t for t in original_terms if t not in {"pesangon", "phk", "pemutusan", "hubungan", "kerja"}}
    if not specific_terms:
        return True
    meta = doc.get("metadata", {})
    haystack = " ".join([
        str(doc.get("text", "")), str(meta.get("citation_text", "")), str(meta.get("bab_title", "")), str(meta.get("bagian_title", "")),
    ]).lower()
    return any(term in haystack for term in specific_terms)


def filter_relevant_context(query: str, docs: List[Dict[str, Any]], max_docs: int = MAX_CONTEXT_DOCS) -> List[Dict[str, Any]]:
    terms = query_terms(expand_query_terms(query))
    scored = []
    for doc in docs:
        if not has_original_specific_overlap(query, doc):
            continue
        score = doc_relevance_score(query, doc)
        overlap = int(round(max(score, 0) * max(len(terms), 1))) if terms else 1
        if (not terms or overlap >= MIN_QUERY_TERM_OVERLAP) and score > 0:
            doc["context_relevance_score"] = score
            scored.append(doc)
    if not scored:
        return docs[:1]
    return sorted(scored, key=lambda d: d.get("context_relevance_score", 0.0), reverse=True)[:max_docs]


def retrieve_context(query: str, k: int = 8, fetch_k: int = 50) -> List[Dict[str, Any]]:
    candidates = []
    seen = set()
    for q in retrieval_queries(query):
        hits = retrieve_documents(q, k=max(k * 6, 18), fetch_k=fetch_k)
        for hit in hits:
            key = hit.get("id") or (
                hit.get("metadata", {}).get("source_file", ""),
                hit.get("metadata", {}).get("pasal_id", ""),
                hit.get("metadata", {}).get("chunk_index", ""),
            )
            if key in seen:
                continue
            seen.add(key)
            candidates.append(hit)
    reranked = rerank_documents(expand_query_terms(query), candidates, k=max(k * 6, 18))
    return filter_relevant_context(query, reranked, max_docs=k)


def build_reference(meta: Dict[str, Any]) -> str:
    citation = meta.get("citation_text") or meta.get("citation") or ""
    source = meta.get("source_file") or meta.get("file_name") or ""
    pasal = meta.get("pasal_id") or meta.get("article") or ""
    return " | ".join([str(p) for p in [citation, source, pasal] if str(p).strip()])


def build_context(docs: List[Dict[str, Any]], max_docs: int | None = None) -> str:
    selected = docs[:max_docs] if max_docs else docs
    blocks = []
    for i, d in enumerate(selected, 1):
        meta = d.get("metadata", {})
        ref = build_reference(meta) or f"Dokumen {i}"
        text = normalize_text(d.get("text", ""))[:2500]
        blocks.append(f"SUMBER HUKUM {i}: {ref}\n{text}")
    return "\n\n".join(blocks)


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = os.getenv("RAG_LLM_MODEL_ID", "Qwen/Qwen3.5-9B")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "700"))
USE_4BIT = os.getenv("USE_4BIT", "1") == "1" and torch.cuda.is_available()

quantization_config = None
if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("model ready", MODEL_ID)

In [ ]:
def strip_thinking(text: str) -> str:
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*(analysis|reasoning)\s*:\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


def sanitize_output(text: str) -> str:
    text = strip_thinking(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def apply_chat_template_no_thinking(messages: List[Dict[str, str]]) -> str:
    if hasattr(processor, "apply_chat_template"):
        try:
            return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return "\n".join([f"{m['role']}: {m['content']}" for m in messages]) + "\nassistant:"


def generate_chat_text(messages: List[Dict[str, str]], max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt = apply_chat_template_no_thinking(messages)
    inputs = processor(text=prompt, return_tensors="pt", padding=True, truncation=True, max_length=12000).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.04,
            top_p=0.9,
            eos_token_id=getattr(processor, "eos_token_id", None),
            pad_token_id=getattr(processor, "eos_token_id", None),
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = processor.decode(new_ids, skip_special_tokens=True)
    return strip_thinking(text)


CONDITIONAL_LOGIC_GUARDRAIL = """
ATURAN KRITIS - PEMISAHAN KONDISI HUKUM (WAJIB DIPATUHI):
Saat dokumen memuat BEBERAPA AYAT dengan kondisi berbeda dalam satu pasal yang sama,
kamu WAJIB mengidentifikasi ayat mana yang berlaku untuk kasus pengguna, lalu HANYA
gunakan konsekuensi hukum dari ayat tersebut. DILARANG KERAS menggabungkan kondisi
dari dua ayat berbeda menjadi satu narasi.

Contoh penerapan pada Pasal 52 PP No. 35 Tahun 2021:
  KONDISI A - Ayat (1): PHK karena pelanggaran ketentuan yang didahului SP ke-1, ke-2,
    dan ke-3. Konsekuensi: pesangon 0,5x, UPMK 1x, UPH. JANGAN sebut ini
    "pelanggaran berat". JANGAN sebut konsekuensi ini untuk kondisi mendesak.
  KONDISI B - Ayat (2)/(4): PHK karena pelanggaran bersifat mendesak (dahulu disebut
    pelanggaran berat). Dilakukan TANPA SP 1,2,3. Konsekuensi: TIDAK BERHAK PESANGON,
    TIDAK BERHAK UPMK, hanya berhak UPH dan Uang Pisah.
  LARANGAN ABSOLUT: Jangan pernah menulis "pelanggaran berat" lalu menyebut
    "pesangon 0,5x" karena itu kontradiksi fatal. Pilih satu kondisi sesuai pertanyaan.

Prinsip ini berlaku umum untuk SEMUA pasal yang memiliki beberapa ayat dengan
kondisi berbeda di seluruh dokumen hukum yang tersedia.
"""

SYSTEM_PROMPT = (
    "Kamu adalah pakar hukum ketenagakerjaan Indonesia yang sangat teliti.\n"
    "Saat membaca dokumen hukum, kamu harus memahami bahwa setiap AYAT memiliki kondisi (syarat) dan konsekuensi yang BERBEDA.\n\n"
    "ATURAN UTAMA:\n"
    "1. JAWAB HANYA berdasarkan KONTEKS. DILARANG mengarang atau memakai pengetahuan luar.\n"
    "2. Langsung sebutkan sumber hukumnya secara natural, contoh: Peraturan Pemerintah No. 35 Tahun 2021, Pasal 52 ayat (2).\n"
    "   Sertakan nomor AYAT jika relevan karena ini penting untuk membedakan kondisi hukum yang berbeda.\n"
    "3. DILARANG memakai kode [R1], [R2], SUMBER HUKUM 1, atau ID internal lain di jawaban.\n"
    "4. DILARANG menulis daftar referensi di dalam jawaban; sistem akan mencetak referensi terpisah.\n"
    "5. PENANGANAN KETERBATASAN INFORMASI:\n"
    "   a. Jika pertanyaan menggunakan istilah lama, bahasa awam, atau sinonim "
    "(contoh: 'pelanggaran berat' = 'pelanggaran bersifat mendesak', "
    "'kontrak' = 'PKWT', 'dipecat' = 'PHK'), JANGAN tolak pertanyaan. "
    "Petakan ke istilah resmi dalam dokumen, sebutkan perubahan istilah tersebut secara singkat di awal jawaban, lalu langsung jawab substansinya.\n"
    "   b. Hanya gunakan kalimat 'Maaf, informasi tersebut tidak tersedia dalam database hukum ketenagakerjaan yang saya miliki.' "
    "jika setelah memetakan semua kemungkinan sinonim pun tidak ada dokumen yang relevan sama sekali.\n"
    "   c. DILARANG KERAS memulai jawaban dengan kata 'Maaf' atau kalimat disclaimer apapun jika konteks sudah tersedia. Langsung jawab substansinya.\n"
    "6. Selalu utamakan aturan terbaru atau aturan yang lebih spesifik jika ada perbedaan antar referensi.\n"
    "7. KETAT PADA KONTEKS: Abaikan dokumen atau pasal yang tidak relevan dengan substansi pertanyaan pengguna.\n"
    "8. SPESIFIK & AKURAT: Sebutkan peraturan, Pasal, beserta AYAT-nya dengan presisi. Jangan pernah mencampuradukkan konsekuensi antar ayat!\n"
    "9. KOMPREHENSIF: Sebutkan semua komponen hak, kewajiban, atau tata cara secara lengkap sesuai ayat yang diekstrak.\n"
    "   Jika berdasarkan pasal tersebut ada ketentuan 'tidak mendapat X', nyatakan pengecualian tersebut dengan tegas.\n"
    "10. Bahasa Indonesia harus rapi, baku, tanpa typo, tanpa campuran aksara asing, tanpa markdown, dan tidak berulang.\n"
    "11. Jika ragu, lebih baik jawab keterbatasan konteks daripada membuat pasal/tahun palsu.\n"
    f"{CONDITIONAL_LOGIC_GUARDRAIL}"
)


def build_messages(question: str, docs: List[Dict[str, Any]], max_context_docs: int = 6) -> List[Dict[str, str]]:
    context = build_context(docs, max_docs=max_context_docs)
    user_prompt = f"KONTEKS REFERENSI HUKUM:\n{context}\n\nPERTANYAAN PENGGUNA:\n{question}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def generate_answer(question: str, k: int = 8, max_context_docs: int = 6) -> Dict[str, Any]:
    t0 = time.time()
    docs = retrieve_context(question, k=k, fetch_k=50)
    answer = sanitize_output(generate_chat_text(build_messages(question, docs, max_context_docs=max_context_docs)))
    latency = time.time() - t0
    refs = []
    for i, d in enumerate(docs, 1):
        meta = d.get("metadata", {})
        refs.append({
            "rank": i,
            "chunk_id": d.get("id", ""),
            "reference": build_reference(meta),
            "source_file": meta.get("source_file", ""),
            "pasal_id": meta.get("pasal_id", ""),
            "rrf_score": d.get("rrf_score", None),
            "rerank_score": d.get("rerank_score", None),
            "context_relevance_score": d.get("context_relevance_score", None),
            "text_preview": normalize_text(d.get("text", ""))[:500],
        })
    return {"question": question, "answer": answer, "references": refs, "latency_seconds": latency}


In [ ]:
question = "Apa hak pekerja jika di-PHK?"
out = generate_answer(question, k=8, max_context_docs=6)

print("PERTANYAAN")
print(question)
print("\nJAWABAN")
print(out["answer"])
print("\nREFERENSI")
for r in out["references"][:6]:
    print(f"{r['rank']}. {r['reference']}")
print(f"\nLatency: {out['latency_seconds']:.2f} detik")

In [ ]:
while True:
    q = input("Tanya RAG hukum (exit untuk berhenti): ").strip()
    if q.lower() in {"exit", "quit", "q"}:
        break
    if not q:
        continue
    out = generate_answer(q, k=8, max_context_docs=6)
    print("\nJAWABAN")
    print(out["answer"])
    print("\nREFERENSI")
    for r in out["references"][:6]:
        print(f"{r['rank']}. {r['reference']}")
    print(f"\nLatency: {out['latency_seconds']:.2f} detik\n")